<a href="https://colab.research.google.com/github/Telop-Auto/telopauto/blob/main/Telop_Auto_v2_0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Telop Auto V2.0
Whisper(音声認識)と Gemini(校正)で、音声ファイルから字幕(SRT)を自動作成します。

制作・著作:Telop Design Lab ／ 不具合や改善点は X の DM までご連絡ください:https://x.com/telopdesignlab

作成した字幕は、Telop Design Lab のテロップテンプレートで Premiere 上でデザインを適用できます:https://terop.booth.pm/

## 使い方
1. 上部メニュー「ランタイム」→「ランタイムのタイプを変更」で **T4 GPU** を選ぶ
2. Gemini API を使用する場合は、APIキーを設定してください(初回のみ)

<details><summary>🔑 Gemini APIキーの設定の仕方(クリックで開く)</summary>

Gemini 校正を使う場合は、APIキーの登録が必要です(Google AI Studio で無料で取得できます)。

1. 左側の 🔑(シークレット)を開く
2. 「新しいシークレットを追加」で、名前に `GEMINI_API_KEY`、値に APIキーを入力
3. 「ノートブックからのアクセス」をオンにする

Gemini 校正を使わない場合は、フォームの「use_gemini」のチェックを外してください。
</details>

3. 下のセルのフォームで、行数・文字数などを設定する(辞書やモデルは、その下の「詳細設定」で変更できます)
4. フォームの左上にある **▶**(丸い再生ボタン)を押す → 数秒で GPU と Gemini APIキーを確認し、**フォームの一番下**にファイル選択ボタンが出るので音声ファイルを選ぶ
5. あとは最後まで自動で進み、SRT がダウンロードされます

**作り直したいとき**は、「音声ファイル」で「前回のファイルで作り直す」を選び、設定を変えてもう一度 ▶ を押してください。アップロードは不要で、変更が必要な工程だけやり直します。エラーで止まった場合も同じ操作で続きから再開できます。


In [ ]:
# @title 🎬 Telop Auto　設定して ▶ を押してください { display-mode: "form" }

# @markdown ### 左にある ▶ を押すと、数秒後にこのフォームの一番下に「ファイル選択」ボタンが出ます

# @markdown ## 音声ファイル
# @markdown ・**新しくアップロード**:ファイルを選んで最初から作成(通常はこれ)<br>
# @markdown ・**前回のファイルで作り直す**:アップロード不要。変えた設定に関係する工程だけやり直します(行数の変更なら数秒)。エラーで止まったときの再開もこれ<br>
# @markdown ・**前回のファイルで最初から作り直す**:アップロード不要。文字起こしから全部やり直します
audio_source = "新しくアップロード"  # @param ["新しくアップロード", "前回のファイルで作り直す", "前回のファイルで最初から作り直す"]

# @markdown ---
# @markdown ## 字幕の形
# @markdown 1行の文字数の目安(推奨:20)
max_chars_per_line = 20  # @param {type:"integer"}
# @markdown 字幕の行数(推奨:2行)
subtitle_lines = "2行"  # @param ["1行", "2行"]
# @markdown 1枚(1ブロック)の最大文字数(2行字幕なら「1行の文字数×2」が目安)
block_max_chars = 40  # @param {type:"integer"}

# @markdown ---
# @markdown ## フィラー削除・校正
# @markdown フィラー削除の強さ(推奨:中)
filler_level = "中"  # @param ["弱", "中", "強"]
# @markdown Gemini で誤字・誤変換を校正する(APIキーが必要)
use_gemini = True  # @param {type:"boolean"}

# ==========================================
# 以下は編集不要
# ==========================================
TELOPAUTO_REF = "v2.0"


def _telopauto_main():
    import importlib.util
    import os
    import subprocess
    import sys

    from IPython.display import HTML, display

    def stop(msg):
        print()
        print("━" * 40)
        print("⛔ " + msg)
        print("━" * 40)

    # ---- 1. GPU(数秒)----
    print("=== 事前チェック ===")
    import torch

    if not torch.cuda.is_available():
        print("⚠️ GPU が使えない状態です。")
        stop("上部メニュー「ランタイム」→「ランタイムのタイプを変更」で「T4 GPU」を選んで保存し、"
             "もう一度 ▶ を押してください(アップロードはまだ不要です)。")
        return
    print(f"✅ GPU:{torch.cuda.get_device_name(0)}")

    # ---- 2. Gemini APIキーの登録確認(数秒)----
    if use_gemini:
        from google.colab import userdata

        key = None
        problem = "読み込めませんでした。少し待ってからもう一度 ▶ を押してください"
        for _ in range(3):
            try:
                key = userdata.get("GEMINI_API_KEY")
                break
            except Exception as e:
                name = type(e).__name__
                if name == "SecretNotFoundError":
                    problem = "登録されていません"
                    break
                if name == "NotebookAccessError":
                    problem = "の「ノートブックからのアクセス」がオフになっています"
                    break
                import time
                time.sleep(3)
        if not key:
            print(f"⚠️ Gemini APIキー(GEMINI_API_KEY)が{problem}。")
            stop("左側の🔑でキーを登録して「ノートブックからのアクセス」をオンにするか、"
                 "「use_gemini」のチェックを外して、もう一度 ▶ を押してください。")
            return
        print("✅ Gemini APIキー:登録を確認しました")
    else:
        print("ℹ️ Gemini校正:オフ")

    # ---- 3. インストール(裏で開始し、アップロード中に進める)----
    # GitHub 上の最新版(コミット)と、入っている版を比べ、違うときだけ入れ直す
    marker = "/content/.telopauto_ref"
    installed = open(marker).read().strip() if os.path.exists(marker) else None
    wanted = TELOPAUTO_REF
    try:
        out = subprocess.run(
            ["git", "ls-remote", "https://github.com/Telop-Auto/telopauto", TELOPAUTO_REF,
             f"refs/tags/{TELOPAUTO_REF}", f"refs/heads/{TELOPAUTO_REF}"],
            capture_output=True, text=True, timeout=15,
        ).stdout.split()
        if out:
            wanted = f"{TELOPAUTO_REF}@{out[0]}"
    except Exception:
        pass
    installer = None
    if importlib.util.find_spec("telopauto") is None or installed != wanted:
        # 初回は必要なライブラリごと、更新時は本体だけを入れ直す
        update_only = ["--force-reinstall", "--no-deps"] if importlib.util.find_spec("telopauto") else []
        installer = subprocess.Popen(
            [sys.executable, "-m", "pip", "install", "-q", *update_only,
             f"git+https://github.com/Telop-Auto/telopauto@{TELOPAUTO_REF}"],
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        )

    # ---- 4. 音声ファイル ----
    last_file_record = "/content/.telopauto_last_file"
    path = None
    if audio_source != "新しくアップロード":
        if os.path.exists(last_file_record):
            candidate = open(last_file_record).read().strip()
            if os.path.exists(candidate):
                path = candidate
        if path:
            print(f"ℹ️ 前回のファイルを使います:{os.path.basename(path)}")
        else:
            print("ℹ️ 前回のファイルが見つかりません(セッションが切れた可能性があります)。アップロードしてください。")
    if not path:
        from google.colab import files

        display(HTML(
            '<div style="margin:12px 0;padding:14px 18px;border:3px solid #f9ab00;border-radius:8px;'
            'background:#fef7e0;color:#222;font-size:18px;font-weight:bold">'
            '👇 下の「ファイル選択」から音声ファイルを選んでください'
            '<div style="font-size:13px;font-weight:normal;margin-top:4px">'
            'MP3 / WAV / M4A / MP4 など。長尺素材は MP3 がおすすめです。</div></div>'
        ))
        uploaded = files.upload()
        names = [n for n in uploaded if not n.startswith(".")]
        if not names:
            stop("音声ファイルが選ばれませんでした。もう一度 ▶ を押してください。")
            return
        if len(names) > 1:
            print(f"※ 複数のファイルが選ばれたため、最初の「{names[0]}」を処理します。")
        path = os.path.abspath(names[0])

    # ---- 5. インストールの完了を待つ ----
    if installer is not None:
        if installer.poll() is None:
            print("準備の完了を待っています...")
        output, _ = installer.communicate()
        if installer.returncode != 0:
            print(output[-2000:])
            stop("準備(インストール)に失敗しました。「前回のファイルで作り直す」を選んでもう一度 ▶ を押してください。")
            return
        with open(marker, "w") as f:
            f.write(wanted)
        # 同じセッションで前の版を読み込み済みでも、新しい版が使われるようにする
        for name in [m for m in sys.modules if m == "telopauto" or m.startswith("telopauto.")]:
            del sys.modules[name]

    # ---- 6. SRT作成 ----
    from telopauto import app
    from telopauto.settings import Settings

    # 詳細設定(下の「詳細設定」セルを実行していなければ既定値)
    detail = globals().get("TELOPAUTO_DETAIL") or {}
    app.run(
        Settings(
            max_chars_per_line=max_chars_per_line,
            lines=1 if subtitle_lines == "1行" else 2,
            block_max_chars=block_max_chars,
            filler_level=filler_level,
            use_gemini=use_gemini,
            **detail,
        ),
        audio_path=path,
        start_over=(audio_source == "前回のファイルで最初から作り直す"),
        detail_is_default=not detail,
    )

_telopauto_main()

## ⚙️ 詳細設定(通常は変更不要・クリックで開く)
ユーザー辞書・Gemini補助辞書・Whisperモデルなどを変更できます。変更したら **このセルの ▶ → 上のメインのセルの ▶** の順に押してください。

In [ ]:
# @title ⚙️ 詳細設定(変更したら、このセルの ▶ を押してから、上のセルの ▶ を押してください) { display-mode: "form" }

# @markdown Whisperモデル(large-v3:高精度 ／ medium:高速・長尺向け)
whisper_model = "large-v3"  # @param ["large-v3", "medium"]
# @markdown 追加で削除するフィラー(カンマ区切り)
custom_fillers = "あのー,えっと,ええと,なんか"  # @param {type:"string"}
# @markdown ユーザー辞書「置換前=置換後」(複数は | で区切る)。登録した語は途中で改行されません
user_dictionary = "プレミア=Premiere | アイフォン=iPhone | 厳選かけ流し=源泉かけ流し"  # @param {type:"string"}
# @markdown Gemini補助辞書:校正時に参考にする固有名詞(カンマ区切り)。登録した語は途中で改行されません
gemini_assist_words = "iPhone,Premiere,Telop Design Lab"  # @param {type:"string"}
# @markdown Geminiモデル名
gemini_model = "gemini-3.5-flash"  # @param {type:"string"}

TELOPAUTO_DETAIL = dict(
    whisper_model=whisper_model,
    custom_fillers=custom_fillers,
    user_dictionary=user_dictionary,
    gemini_assist_words=gemini_assist_words,
    gemini_model=gemini_model,
)
print("✅ 詳細設定を反映しました。上のセルの ▶ を押すと、この設定で作成します。")

## 注意事項
- **長尺素材は MP3 を推奨します**(WAV はアップロードに時間がかかることがあります)。動画ファイル(MP4 など)もそのまま使えます。
- 本ツールは継続的に改善・アップデートを行っているβ版です。文字起こしや校正の結果には誤りが含まれる場合があります。最終的な内容は必ず確認してください。
- Gemini API には利用制限があります。上限に達した場合、残りの文は校正せずに SRT を作成します。時間をおいて「前回のファイルで作り直す」を選んで ▶ を押すと、未校正の文だけを校正します。
- Google Colab の無料 GPU には利用上限があります。
- 本ノートブックは個人・業務用途を問わずご利用いただけます。ただし、本ノートブックおよびその複製・改変物の再配布・再販売は禁止します。